# Advanced Tracing & RunConfig

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to build custom trace processors, redact sensitive data from traces, use `RunConfig` for model input filtering and error handling, control `max_turns`, and clone agents with `agent.clone()`.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Custom Trace Processor

Create a custom trace processor to send traces to your own systems.

In [ ]:
from agents.tracing import add_trace_processor, TracingProcessor, Trace, Span


class LoggingTraceProcessor(TracingProcessor):
    def on_trace_start(self, trace: Trace) -> None:
        print(f"[TRACE START] {trace.trace_id} - {trace.name}")

    def on_trace_end(self, trace: Trace) -> None:
        print(f"[TRACE END] {trace.trace_id}")

    def on_span_start(self, span: Span) -> None:
        print(f"  [SPAN START] {span.span_id} - {span.span_data}")

    def on_span_end(self, span: Span) -> None:
        print(f"  [SPAN END] {span.span_id}")


add_trace_processor(LoggingTraceProcessor())
print("Custom trace processor registered.")

## 4. Run Agent with Custom Tracing

Run an agent and observe the custom trace output.

In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Traced Agent",
    instructions="You are a helpful assistant.",
)

result = await Runner.run(agent, "What is the speed of light?")
print(f"\nOutput: {result.final_output}")

## 5. Replace All Trace Processors

Use `set_trace_processors` to replace the default OpenAI exporter entirely.

In [ ]:
from agents.tracing import set_trace_processors

set_trace_processors([LoggingTraceProcessor()])

result = await Runner.run(agent, "What is gravity?")
print(f"\nOutput: {result.final_output}")

## 6. Sensitive Data Redaction

Set `trace_include_sensitive_data=False` to strip PII from traces.

In [ ]:
from agents import RunConfig

result = await Runner.run(
    agent,
    "Process this customer's credit card ending in 4242",
    run_config=RunConfig(trace_include_sensitive_data=False),
)
print(result.final_output)

## 7. Model Input Filtering with call_model_input_filter

Inspect or modify messages before each LLM call.

In [ ]:
def filter_model_input(messages):
    """Log and redact CONFIDENTIAL markers before they reach the model."""
    print(f"Sending {len(messages)} messages to model")
    for msg in messages:
        if hasattr(msg, "content") and isinstance(msg.content, str):
            if "CONFIDENTIAL" in msg.content:
                msg.content = msg.content.replace("CONFIDENTIAL", "[REDACTED]")
    return messages


config = RunConfig(
    call_model_input_filter=filter_model_input,
)

result = await Runner.run(agent, "Summarize this CONFIDENTIAL report about Q4 earnings", run_config=config)
print(result.final_output)

## 8. Error Handlers and max_turns

Use `error_handlers` in `RunConfig` to gracefully handle max_turns exceeded.

In [ ]:
from agents import function_tool


def handle_max_turns(error):
    print(f"Agent exceeded max turns: {error}")
    return "I wasn't able to complete the task within the allowed steps. Please try a simpler request."


@function_tool
def research(topic: str) -> str:
    """Research a topic."""
    return f"Research results for: {topic}"


research_agent = Agent(
    name="Researcher",
    instructions="Research topics thoroughly. Use the research tool multiple times if needed.",
    tools=[research],
)

config = RunConfig(
    max_turns=3,
    error_handlers={"max_turns": handle_max_turns},
)

result = await Runner.run(
    research_agent,
    "Research quantum computing, AI, and blockchain",
    run_config=config,
)
print(result.final_output)

## 9. Agent Cloning with agent.clone()

Create modified copies of an agent without mutating the original.

In [ ]:
base_agent = Agent(
    name="Base Agent",
    instructions="You are a helpful assistant.",
    model="gpt-4o",
)

fast_clone = base_agent.clone(
    name="Fast Agent",
    model="gpt-4o-mini",
)

verbose_clone = base_agent.clone(
    instructions="You are a helpful assistant. Provide detailed, thorough answers with examples.",
)

print(f"Base: {base_agent.name}, model={base_agent.model}")
print(f"Fast: {fast_clone.name}, model={fast_clone.model}")
print(f"Verbose: {verbose_clone.name}, instructions={verbose_clone.instructions[:60]}...")

## 10. Production RunConfig

Combine tracing, redaction, input filtering, and error handling in a single configuration.

In [ ]:
production_config = RunConfig(
    model="gpt-4o",
    max_turns=10,
    trace_include_sensitive_data=False,
    call_model_input_filter=filter_model_input,
    error_handlers={"max_turns": handle_max_turns},
    max_retries=3,
    retry_delay=1.0,
)

production_agent = Agent(
    name="Production Agent",
    instructions="You are a production assistant.",
)

result = await Runner.run(production_agent, "Help me with my account", run_config=production_config)
print(result.final_output)

## Key Takeaways

- Use `add_trace_processor` to send traces to custom backends alongside the default exporter
- Use `set_trace_processors` to replace the default exporter entirely
- Set `trace_include_sensitive_data=False` in `RunConfig` to strip PII from traces
- Use `call_model_input_filter` to inspect or modify messages before each LLM call
- Configure `error_handlers` in `RunConfig` for graceful max_turns handling
- Clone agents with `agent.clone()` for variant creation without mutating the original